# Notebook reactivity with ipywidgets

A swiftmap `Map` **is** an ipywidget (anywidget builds on ipywidgets), so
everything ipywidgets does — `observe`, `interact`, layout containers, trait
links — works with it directly, no server involved. This notebook mirrors the
patterns the Shiny apps use and doubles as an integration probe: a findings
section at the end says honestly what is smooth and what is manual.

Needs `swiftmap`, `pandas`, and `ipywidgets`.

In [ ]:
import numpy as np
import pandas as pd
import ipywidgets as widgets
from swiftmap import Map

rng = np.random.default_rng(7)
n = 200
df = pd.DataFrame({
    "lat": 36.02 + rng.normal(0, 0.06, n),
    "lon": -5.45 + rng.normal(0, 0.10, n),
    "site": [f"Sensor {i:03d}" for i in range(n)],
    "reading": np.round(rng.gamma(4, 4, n), 1),
    "status": rng.choice(["Active", "Idle", "Fault"], n, p=[0.6, 0.3, 0.1]),
})

m = Map()
m.add_circle_markers(df, name="Sensors",
                     layer_group=["Sensors", "status"],
                     color_col="reading")
m.configure_group("Sensors", collapsed=False)
m

## The build-once rule, notebook edition

In Shiny, `@render_widget` can silently rebuild the map — which is why
`map_effect` exists. In a notebook the problem disappears: **you hold the live
object**. Every cell below mutates `m` in place, and the map above updates as you
run them — each change travels as a small patch, never a rebuild.

## Widget → map: `observe`

Shiny's

```python
@map_effect(mapview, event=input.status)
def filter_status(m): ...
```

becomes `dropdown.observe(handler, names="value")`. Two lines of ceremony
`map_effect` absorbs in Shiny are yours here: unpack `change["new"]`, and open
`m.batch()` so several updates leave as one message. (The third — skipping until
the widget renders — is moot: `m` exists from the moment you built it.)

In [ ]:
status_dd = widgets.Dropdown(options=["All", "Active", "Idle", "Fault"],
                             description="Status")

def filter_status(change):
    wanted = change["new"]
    with m.batch():
        if wanted == "All":
            m.select(None, scope="Sensors")
        else:
            m.select(m.find_layers(group=f"Sensors/{wanted}"), scope="Sensors")

status_dd.observe(filter_status, names="value")
status_dd

## Map → Python: the map's traits are observable too

`clicked_layer_id` and `selected_index` are plain traitlets — and `click_seq`
bumps on **every** click, feature or open map, precisely so observers never miss
one (the other traits only fire when their value changes). A click on open map
clears the feature traits and reports `clicked_latlng` as `[lat, lon]`. Observe
`click_seq`, read the rest:

In [ ]:
click_out = widgets.HTML("<i>Click a sensor — or open map — above.</i>")

def on_click(change):
    layer = m.get_layer(m.clicked_layer_id)
    if layer is None:
        lat, lon = m.clicked_latlng
        click_out.value = f"open map at {lat:.4f}, {lon:.4f}"
    else:
        click_out.value = (f"<b>{layer.get('layer_group')}</b> / "
                           f"{layer.get('name')} — feature {m.selected_index}")

m.observe(on_click, names="click_seq")
click_out

## Feature spotlight from a slider

`set_feature_styles` indexes features **within one layer**, so this section uses
a flat single-layer map where DataFrame row and feature index correspond — the
same reason the linked-table Shiny app gives each dwell its own layer instead of
fanning one layer out into folders.

In [ ]:
m2 = Map()
m2.add_circle_markers(df, name="Readings", color_col="reading")

threshold = widgets.IntSlider(value=0, min=0, max=40, description="Spotlight ≥")

def spotlight(change):
    t = change["new"]
    if t == 0:
        m2.set_feature_styles("Readings", {})
        return
    loud = {i: {"color": "#ffcc00", "radius": 13}
            for i, r in enumerate(df["reading"]) if r >= t}
    m2.set_feature_styles("Readings", loud)

threshold.observe(spotlight, names="value")
widgets.VBox([threshold, m2])

## `interact`, the one-liner

`interact` re-*calls* the function on every change — safe here because the
function mutates the live map and never rebuilds it. Fitting the view to a folder,
driven by a generated dropdown:

In [ ]:
from ipywidgets import interact

def zoom_to(status="Active"):
    m.fit_bounds(m.bounds_of(group=f"Sensors/{status}"), padding=30)

interact(zoom_to, status=["Active", "Idle", "Fault"]);

## Two-way time sync

`time_current` (epoch ms) is a trait like any other: observe it to follow
playback from Python, set it to move the slider. Scroll up — the slider appears
on the first map.

In [ ]:
import datetime

pings = pd.DataFrame({
    "lat": 36.00 + np.cumsum(rng.normal(0.002, 0.003, 48)),
    "lon": -5.80 + np.cumsum(rng.normal(0.009, 0.005, 48)),
    "timestamp": pd.date_range("2026-08-01", periods=48, freq="30min", tz="UTC"),
})
m.add_circle_markers(pings, name="Pings", layer_group="Tracks",
                     color="#f28e2b")
m.make_time_layer("Pings", period="PT1H")

tick_label = widgets.Label("(press play on the map's slider)")

def show_tick(change):
    stamp = datetime.datetime.fromtimestamp(change["new"] / 1000,
                                            tz=datetime.timezone.utc)
    tick_label.value = f"slider at {stamp:%Y-%m-%d %H:%M} UTC"

m.observe(show_tick, names="time_current")

noon = widgets.Button(description="Jump to noon")
noon.on_click(lambda _btn: setattr(
    m, "time_current",
    datetime.datetime(2026, 8, 1, 12, tzinfo=datetime.timezone.utc)
    .timestamp() * 1000))

widgets.HBox([noon, tick_label])

## Sizing and layout

The map is a DOMWidget: it composes into `VBox`/`HBox`/`Tab`/`AppLayout` (the
spotlight section above put one in a `VBox`), and `height` is a first-class
synced trait — assigning it resizes a live map:

In [ ]:
m2.height = "500px"

> `Map(height="500px")` works at construction too, and the generic ipywidgets
> route (`m2.layout.height`) still works alongside.

Displaying the same `Map` object twice gives two synced views of one model —
toggle a sidebar checkbox in either and both follow. Useful for a controls-beside-
map layout without duplicating any data.

## Findings: what this probe surfaced

Mirroring the Shiny apps in plain ipywidgets turned up **no blockers** — and the
two papercuts the first pass of this notebook found were fixed the same day,
which is the probe doing exactly its job:

- `observe`, `interact`, `on_click`, layout containers, and trait observation on
  the map itself all work as ipywidgets promises. Patches keep multi-view and
  mutation costs small, same as under Shiny.
- `Map(height=)` was accepted-but-unwired when this notebook first ran; it is
  now a synced trait, used above. Repeat clicks used to be invisible to
  observers; `click_seq` now exists for exactly that, also used above.
- The boilerplate delta against Shiny's `map_effect` is ~3 lines per control:
  unpack `change["new"]`, open `m.batch()`, wire the `observe`. A notebook
  helper mirroring `map_effect` would be sugar, not capability — worth deciding
  after real notebook use, not before.